# FULL DATASET ANALYSIS

This notebook contains comprehensive analysis for the full datasets

**Datasets:**
- train_full: 5,337,414 rows, 94 columns
- test_full: 1,447,107 rows, 92 columns

**Analysis Pipeline:**
1. Data identification and structure
2. Missing values analysis
3. Distribution analysis (all types)
4. Outlier detection
5. Data type optimization
6. Standardization strategies

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import polars as pl
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
import time

# Display configuration - show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Set up paths
base_dir = Path("..")
full_train_path = base_dir / "data" / "train.parquet"
full_test_path = base_dir / "data" / "test.parquet"

print("✅ Imports and configuration complete")

✅ Imports and configuration complete


In [3]:
# ============================================
# LOAD FULL DATASETS
# ============================================
print("="*60)
print("LOADING FULL DATASETS")
print("="*60)

# Load full datasets
train_full = pd.read_parquet(full_train_path)
test_full = pd.read_parquet(full_test_path)

print(f"Train full: {train_full.shape}")
print(f"Test full: {test_full.shape}")

print(f"\n✅ Full datasets loaded")
print(f"   Train: {train_full.shape}")
print(f"   Test: {test_full.shape}")

LOADING FULL DATASETS
Train full: (5337414, 94)
Test full: (1447107, 92)

✅ Full datasets loaded
   Train: (5337414, 94)
   Test: (1447107, 92)


## STEP 1: DATA IDENTIFICATION AND STRUCTURE

In [2]:
# ============================================
# BASIC DATA STRUCTURE ANALYSIS
# ============================================
print("="*60)
print("DATA STRUCTURE ANALYSIS")
print("="*60)

# Train dataset structure
print("\n📋 TRAIN DATASET STRUCTURE:")
print(f"Shape: {train_full.shape}")
print(f"Columns: {train_full.columns.tolist()}")
print(f"\nData types:")
print(train_full.dtypes.value_counts())

# Test dataset structure  
print("\n📋 TEST DATASET STRUCTURE:")
print(f"Shape: {test_full.shape}")
print(f"Columns: {test_full.columns.tolist()}")
print(f"\nData types:")
print(test_full.dtypes.value_counts())

# Column differences
train_cols = set(train_full.columns)
test_cols = set(test_full.columns)
only_in_train = train_cols - test_cols
only_in_test = test_cols - train_cols

print(f"\n📊 COLUMN DIFFERENCES:")
print(f"Only in train: {only_in_train}")
print(f"Only in test: {only_in_test}")
print(f"Common columns: {len(train_cols & test_cols)}")

DATA STRUCTURE ANALYSIS

📋 TRAIN DATASET STRUCTURE:


NameError: name 'train_full' is not defined

In [4]:
# ============================================
# DETAILED COLUMN INFORMATION
# ============================================
print("="*60)
print("DETAILED COLUMN ANALYSIS")
print("="*60)

# Categorical columns analysis
cat_cols = ['code', 'sub_code', 'sub_category']
print("\n🏷️ CATEGORICAL COLUMNS:")
for col in cat_cols:
    if col in train_full.columns:
        unique_count = train_full[col].nunique()
        print(f"\n{col}:")
        print(f"  Unique values: {unique_count}")
        print(f"  Sample values: {train_full[col].unique()[:10].tolist()}")
        if col == 'sub_category':
            print(f"  Value counts:")
            print(train_full[col].value_counts())

# Temporal columns
temporal_cols = ['ts_index', 'horizon']
print("\n⏰ TEMPORAL COLUMNS:")
for col in temporal_cols:
    if col in train_full.columns:
        print(f"\n{col}:")
        print(f"  Min: {train_full[col].min()}")
        print(f"  Max: {train_full[col].max()}")
        print(f"  Unique values: {train_full[col].nunique()}")

# Target and weight
special_cols = ['y_target', 'weight']
print("\n🎯 SPECIAL COLUMNS:")
for col in special_cols:
    if col in train_full.columns:
        print(f"\n{col}:")
        print(f"  Type: {train_full[col].dtype}")
        print(f"  Min: {train_full[col].min()}")
        print(f"  Max: {train_full[col].max()}")
        print(f"  Mean: {train_full[col].mean():.6f}")
        print(f"  Std: {train_full[col].std():.6f}")

DETAILED COLUMN ANALYSIS

🏷️ CATEGORICAL COLUMNS:

code:
  Unique values: 23
  Sample values: ['W2MW3G2L', 'OSJL3A7Y', '660DZME0', '2RBMUWP1', 'QAQDDTPJ', 'W4S29LF4', '10BAVIDU', 'MLAAMU3K', 'K7Y1TTAH', 'VFWIFJPS']

sub_code:
  Unique values: 180
  Sample values: ['J0G2B0KU', 'QCLPKYBU', 'BHYOOVXQ', 'P9ZIZD7U', '6TA07NOO', 'VYQLNVSJ', 'RZZAJL41', 'JFGVYNGP', 'YGQ34KOD', 'BZRO0RKW']

sub_category:
  Unique values: 5
  Sample values: ['PZ9S1Z4V', 'V8BKY1IV', 'DPPUO5X2', 'NQ58FVQM', 'PHHHVYZI']
  Value counts:
sub_category
PZ9S1Z4V    1074239
DPPUO5X2    1072705
NQ58FVQM    1067164
PHHHVYZI    1067164
V8BKY1IV    1056142
Name: count, dtype: int64

⏰ TEMPORAL COLUMNS:

ts_index:
  Min: 1
  Max: 3601
  Unique values: 3601

horizon:
  Min: 1
  Max: 25
  Unique values: 4

🎯 SPECIAL COLUMNS:

y_target:
  Type: float64
  Min: -2201.8815779044935
  Max: 2314.4111524982895
  Mean: -0.665905
  Std: 32.527642

weight:
  Type: float64
  Min: 0.0
  Max: 13912217783333.135
  Mean: 16427879.645292
  St